In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import date, datetime, timedelta
from pyspark.sql.window import Window

catalog = "case"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Schema {silver_schema} garantido. Iniciando transformações...")

In [0]:
df_bronze_motivos = spark.table(f"{bronze_schema}.tb_base_motivos")

# display(df_bronze_motivos)

### Tratamento da tabela `base_motivos`

Esta etapa padroniza as colunas `criticidade` e `categoria`, corrigindo os problemas encontrados no diagnóstico da camada bronze.

---

#### 1. Padronização da `criticidade`

**Problema:** o mesmo nível de criticidade aparecia escrito de formas diferentes (`Alta` e `alta`). Em agrupamentos e filtros, essas variações seriam tratadas como níveis distintos.

**Tratamento:** os espaços nas pontas são removidos (`trim`) e o texto é convertido para minúsculo (`lower`).

| Antes | Depois |
|---|---|
| `Alta` / `alta` | `alta` |
| `Media` | `media` |
| `baixa` | `baixa` |

**Resultado:** a coluna passa a ter apenas três valores: `alta`, `media` e `baixa`.

---

#### 2. Preenchimento e padronização da `categoria`

**Problema:** três motivos estavam sem categoria (ids 12, 14 e 16). O valor vazio pode chegar como `null` ou como texto vazio (`""`), dependendo da leitura do CSV, por isso os dois casos são tratados.

**Tratamento:**
- Categorias nulas ou vazias recebem o valor `Outro`.
- As demais têm os espaços nas pontas removidos.
- Todo o resultado é convertido para minúsculo, seguindo o mesmo padrão da `criticidade`.

| id_motivo | nome_motivo | Antes | Depois |
|---|---|---|---|
| 12 | Contratação de cartão adicional | *(vazio)* | `outro` |
| 14 | Transferência de agência | *(vazio)* | `outro` |
| 16 | Cancelamento de conta | *(vazio)* | `outro` |

**Resultado:** a coluna passa a ter apenas os cinco valores definidos no PDF do case: `cartao`, `financeiro`, `beneficios`, `atendimento` e `cadastral`.

**Correção confirmada pelo PDF** (documento *Dinâmica de Solução de Case — V-Credit*, seção 4 — Base de Motivos):

| id_motivo | nome_motivo | Antes (Silver v1) | Agora (Silver v2) | Justificativa (PDF) |
|---|---|---|---|---|
| 12 | Contratação de cartão adicional | `outro` | `cartao` | *"Cartão: bloqueio, desbloqueio, **contratações** e compras"* |
| 14 | Transferência de agência | `outro` | `cadastral` | *"Cadastral: telefone, e-mail, **agência** etc."* |
| 16 | Cancelamento de conta | `outro` | `financeiro` | *"Financeiro: faturas, limites, **contratos**, pagamentos e renegociações"* |

Os três motivos que chegavam sem categoria na Bronze agora são mapeados diretamente por `id_motivo` antes do fallback genérico, que permanece como rede de segurança para futuros motivos sem categoria.

---

#### 3. Normalização do `nome_motivo`

**Motivo:** a coluna `motivo` da `tb_chamados` já foi normalizada na Silver (lowercase, sem acentos). A `tb_motivos` preserva o `nome_motivo` original (com acentos e capitalização), o que força um `translate` em todo join entre as duas tabelas.

**Tratamento:** adicionar a coluna `nome_motivo_normalizado` aplicando a mesma transformação: `lower` + `translate` de acentos para sem-acentos.

| `nome_motivo` (original) | `nome_motivo_normalizado` |
|---|---|
| Consulta de fatura | `consulta de fatura` |
| Bloqueio de cartão | `bloqueio de cartao` |
| Contratação de cartão adicional | `contratacao de cartao adicional` |
| Transferência de agência | `transferencia de agencia` |

**Resultado:** o join no Gold passa a ser direto: `tb_chamados.motivo = tb_motivos.nome_motivo_normalizado`.

In [0]:
#Padroniza em minusculo
df_silver_motivos = df_bronze_motivos.withColumn("criticidade", F.lower(F.trim(F.col("criticidade"))))

# Mapeamento de categorias conforme definido no PDF do case (seção 4 — Base de Motivos):
#   Financeiro: faturas, limites, contratos, pagamentos e renegociações
#   Cartão: bloqueio, desbloqueio, contratações e compras
#   Cadastral: telefone, e-mail, agência etc.
#   Atendimento: problemas técnicos nos canais digitais (site, app, chatbot, URA)
#   Benefícios: programas de pontos e benefícios
#
# Os ids 12, 14 e 16 chegavam sem categoria na Bronze. O PDF confirma:
#   id 12 (Contratação de cartão adicional) → cartao  ("bloqueio, desbloqueio, contratações e compras")
#   id 14 (Transferência de agência)          → cadastral ("telefone, e-mail, agência etc.")
#   id 16 (Cancelamento de conta)             → financeiro ("faturas, limites, contratos, pagamentos e renegociações")
# O fallback "outro" permanece como rede de segurança para futuros motivos sem categoria.
df_silver_motivos = df_silver_motivos.withColumn(
    "categoria",
    F.lower(
        F.when(F.col("id_motivo") == 12, F.lit("cartao"))
         .when(F.col("id_motivo") == 14, F.lit("cadastral"))
         .when(F.col("id_motivo") == 16, F.lit("financeiro"))
         .when(F.col("categoria").isNull() | (F.trim(F.col("categoria")) == ""), F.lit("outro"))
         .otherwise(F.trim(F.col("categoria")))
    )
)

# Adiciona nome_motivo normalizado (lowercase sem acentos) para permitir join direto
# com tb_chamados.motivo, que já foi normalizado da mesma forma na Silver.
# Mesma transformação: lower + translate de acentos → sem acento.
df_silver_motivos = df_silver_motivos.withColumn(
    "nome_motivo_normalizado",
    F.lower(
        F.translate(
            F.col("nome_motivo"),
            "áàâãéêíóôõúçÁÀÂÃÉÊÍÓÔÕÚÇ",
            "aaaaeeioooucaaaaeeiooouc"
        )
    )
)

display(df_silver_motivos)

In [0]:
df_silver_motivos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_motivos")


print(f"Tabela {silver_schema}.tb_motivos criada/atualizada com sucesso!")
print(f"Bronze (antes do dedup): {df_bronze_motivos.count():,} linhas")
print(f"Silver (após dedup e tratamento): {df_silver_motivos.count():,} linhas")